In [ ]:
# 1. 외부 모듈 자동 새로고침 설정 (loader.py 수정 시 즉각 반영)
%load_ext autoreload
%autoreload 2

# 2. 필수 라이브러리 임포트
import os
import json
import pandas as pd
import FinanceDataReader as fdr
import pykrx
import OpenDartReader
import matplotlib
import seaborn
import scipy
from datetime import date

# 3. 직접 만든 로컬 모듈 임포트
from data.loader import QuantDataLoader

print("✅ 환경 설정 및 전체 라이브러리 정상 로드 완료!")

In [ ]:
import pickle

# 디버깅 분석을 위해 저장해둔 history 데이터 로드
try:
    with open('debug_history.pkl', 'rb') as f:
        history = pickle.load(f)
    print("✅ 'debug_history.pkl' 로드 완료! 이제 디버깅을 시작할 수 있습니다.")
except FileNotFoundError:
    print("❌ 'debug_history.pkl' 파일을 찾을 수 없습니다. 2번 노트북을 먼저 실행해주세요.")

print("==================================================")
print("🔍 파이프라인 탈락률(Funnel) 분석")
print("==================================================\n")

# 각 단계별 생존 종목 수 확인
stages = ['stage1', 'stage2', 'stage3', 'stage4', 'stage5']
previous_count = None

for stage in stages:
    if stage in history:
        current_count = len(history[stage])
        
        # 탈락률 계산
        if previous_count is not None and previous_count > 0:
            drop_rate = (previous_count - current_count) / previous_count * 100
            print(f"📉 {stage} 생존: {current_count}개 (직전 단계 대비 -{drop_rate:.1f}% 탈락)")
        else:
            print(f"🎯 {stage} 생존: {current_count}개 (섹터 통과 후 남은 전체 유니버스)")
            
        previous_count = current_count

# 만약 Stage 4까지 살아남았는데 Stage 5에서 다 죽었다면, 
# Stage 4의 생존자들을 살펴봅니다.
if 'stage4' in history and not history['stage4'].empty:
    print("\n💡 [마지막 생존자] Stage 4 통과 종목 (이들이 Stage 5에서 탈락함)")
    display(history['stage4'][['ticker', 'sector', 'roe', 'pbr']])

In [ ]:
import pandas as pd

print("==================================================")
print("🩺 [정밀 진단] Stage 3 탈락 종목 Raw Data 분석")
print("==================================================\n")

try:
    # 1. Stage 2 통과자 중 Stage 3에서 떨어진 종목 추출
    # history 딕셔너리에 데이터가 남아있다는 가정 하에 진행
    stage2_passed = history['stage2']
    stage3_passed_tickers = history['stage3']['ticker'].tolist() if not history['stage3'].empty else []
    
    stage3_failed_df = stage2_passed[~stage2_passed['ticker'].isin(stage3_passed_tickers)]
    sample_failures = stage3_failed_df.head(5)['ticker'].tolist()
    
    print(f"총 {len(stage3_failed_df)}개 종목이 Stage 3에서 탈락했습니다. 샘플 5개 진단 시작...\n")
    
    # 2. 로더를 직접 호출하여 데이터 계산 흐름 확인
    for ticker in sample_failures:
        print(f"▶️ 대상 종목: {ticker}")
        
        # 최근 6개 분기 시계열 로드
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        
        if len(q_series) < 6:
            print("  ❌ [사유] DATA_TOO_SHORT (가용 데이터 부족)\n")
            continue
            
        # 값 출력 (최근 3개 분기(t=0, 1, 2)와 전년 동기(t=4, 5, 6) 데이터 흐름 확인)
        for i, q in enumerate(q_series[:4]):  # 디버깅을 위해 최근 4분기만 출력
            rev = q.get('revenue', 0)
            sga = q.get('sga', 0)
            print(f"  [t-{i} 분기] 매출액: {rev:,.0f} | 판관비: {sga:,.0f}")
            
            if rev < 0 or sga < 0:
                print("  🚨 [경고] 음수 값 발견! 차분 로직(thstrm_add_amount) 오류 확실시 됨.")
                
        print("-" * 50)
        
except Exception as e:
    print(f"❌ 진단 중 에러 발생: {e}")

In [ ]:
import pandas as pd
from collections import Counter

print("==================================================")
print("🩺 Stage 3 진입 종목(75개) 전수 데이터 상태 판별 (로더 직접 호출)")
print("==================================================\n")

try:
    stage2_passed_df = history.get('stage2')
    if stage2_passed_df is None or stage2_passed_df.empty:
        print("Stage 2 통과 종목이 없습니다.")
    else:
        status_counter = Counter()
        
        print("⏳ 로더(loader)를 통해 75개 종목의 시계열 상태 직접 확인 중...")
        for _, row in stage2_passed_df.iterrows():
            ticker = row['ticker']
            
            # loader를 직접 호출하여 6분기 시계열 확보 시도 (에러 없이 가져오는지 확인)
            q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
            
            if len(q_series) < 6:
                status_counter['DATA_TOO_SHORT (6분기 미달)'] += 1
            else:
                # 데이터가 6개 다 있다면, 음수나 결측치가 있는지 추가 확인
                has_error = False
                for q in q_series:
                    rev = q.get('revenue', 0)
                    sga = q.get('sga', 0)
                    # 데이터가 없거나(NaN), 음수값이 튀어나온 경우(단독값 파싱 에러 의심)
                    if pd.isna(rev) or pd.isna(sga) or rev < 0 or sga < 0:
                        has_error = True
                        break
                        
                if has_error:
                    status_counter['DATA_INVALID (음수 또는 결측치 포함)'] += 1
                else:
                    status_counter['COMPUTED (정상 계산 가능)'] += 1
            
        print("\n📊 [전수조사 결과: 데이터 상태 분포]")
        total = len(stage2_passed_df)
        for status, count in status_counter.items():
            print(f" - {status}: {count}개 종목 ({(count/total)*100:.1f}%)")

except Exception as e:
    print(f"❌ 진단 중 에러 발생: {e}")

In [ ]:
import pandas as pd

print("==================================================")
print("🕵️‍♂️ [심층 추적] 12개 비정상 데이터 원본 파싱 상태 점검")
print("==================================================\n")

try:
    stage2_passed_df = history['stage2']
    invalid_tickers = []
    
    for _, row in stage2_passed_df.iterrows():
        ticker = row['ticker']
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        
        # 6분기 데이터가 모두 존재하는 경우에만 값 검증
        if len(q_series) == 6:
            for q in q_series:
                rev = q.get('revenue', 0)
                sga = q.get('sga', 0)
                # 성장률(변동량)이 아닌, '매출액/판관비 절대치' 자체가 음수인 경우를 적발
                if pd.isna(rev) or pd.isna(sga) or rev < 0 or sga < 0:
                    invalid_tickers.append(ticker)
                    break
    
    print(f"발견된 의심 종목 수: {len(invalid_tickers)}개\n")
    
    # 비정상 데이터 상세 출력하여 원본 값 확인
    for ticker in invalid_tickers:
        print(f"▶️ 종목: {ticker}")
        q_series = loader.get_quarterly_financials_series(ticker, base_date, n_quarters=6)
        for i, q in enumerate(q_series):
            rev_val = q.get('revenue')
            sga_val = q.get('sga')
            
            rev_str = f"{rev_val:>20,}" if pd.notna(rev_val) else f"{'NaN':>20}"
            sga_str = f"{sga_val:>20,}" if pd.notna(sga_val) else f"{'NaN':>20}"
            
            print(f"  [t-{i}] 매출액: {rev_str} | 판관비: {sga_str}")
        print("-" * 50)
        
except Exception as e:
    print(f"❌ 에러: {e}")

In [ ]:
import os
import pandas as pd
from data.loader import QuantDataLoader

def scan_invalid_tickers_accounts():
    loader = QuantDataLoader(use_cache=True)
    
    # 🚨 여기에 DATA_INVALID 판정을 받은 13개 종목의 티커(문자열)를 입력하세요.
    invalid_tickers = ['103140', '069620', '003090', '032350', '185750',
                       '079160', '039130', '016590', '002310', '016800',
                       '037560', '032560', '019680'] # 예시 티커
    
    year = 2023
    reprt_code = '11011' # 사업보고서(11011) 등 결측치가 발생한 분기를 기준
    
    for ticker in invalid_tickers:
        print(f"\n{'='*60}")
        print(f"🔍 [{ticker}] {year}년 손익계산서(IS/CIS) 원본 계정명")
        print(f"{'='*60}")
        
        # 연결재무제표(CFS) 로드
        df = loader.get_financial_statements(ticker, year, reprt_code, fs_div='CFS')
        
        if df is None or df.empty:
            print("캐시된 데이터가 없거나 로드에 실패했습니다.")
            continue
            
        # 손익계산서 항목만 필터링
        is_df = df[df['sj_div'].isin(['IS', 'CIS'])]
        
        if is_df.empty:
            print("손익계산서(IS/CIS) 데이터가 존재하지 않습니다.")
            continue
            
        # 파악을 돕기 위해 계정명, 계정 ID, 당기 금액을 함께 출력
        display_cols = ['account_nm', 'account_id', 'thstrm_amount']
        
        # 판다스 출력 설정 (잘림 방지)
        pd.set_option('display.max_rows', None)
        output = is_df[display_cols].fillna('NaN')
        
        print(output.to_string(index=False))

if __name__ == "__main__":
    scan_invalid_tickers_accounts()

In [ ]:
# 파이프라인 실행 완료 후 (final_df, history = pipeline.run(...) 이후)

target_ticker = '103140'  # 테스트할 종목 코드 (판관비가 없어 NOT_COMPUTABLE이 예상되는 종목)

# 1. Stage 2 통과 여부 확인
stage2_df = history.get('stage2', pd.DataFrame())

if not stage2_df.empty and target_ticker in stage2_df['ticker'].values:
    print(f"✅ [{target_ticker}] Stage 2 생존 확인")
    
    # 2. Stage 3 통과 여부 및 데이터 확인
    stage3_df = history.get('stage3', pd.DataFrame())
    
    if not stage3_df.empty and target_ticker in stage3_df['ticker'].values:
        print(f"✅ [{target_ticker}] Stage 3 통과 (Exempt 구제 로직 정상 작동!)")
        
        # 실제 어떤 값으로 채워져서 통과했는지 해당 행 출력
        target_row = stage3_df[stage3_df['ticker'] == target_ticker]
        print("\n[Stage 3 데이터 확인]")
        print(target_row)
        
    else:
        print(f"🚨 [{target_ticker}] Stage 3에서 탈락(FAILED)했습니다.")
        print("결론: NOT_COMPUTABLE 구제 로직이 제대로 타지 않고 실격 처리 중입니다. Stage 3 코드 수정이 필요합니다.")
else:
    print(f"⚠️ [{target_ticker}] Stage 2에서 이미 탈락하여 Stage 3 검증이 불가능합니다. Stage 2를 통과한 다른 금융/지주사 종목으로 변경해주세요.")

In [ ]:
import yaml
import pandas as pd
from datetime import date
from IPython.display import display  # 주피터 환경용 깔끔한 출력
from backtest.forward_return import BacktestEngine
from core.pipeline import QuantPipeline 
from data.loader import QuantDataLoader

# 판다스 열(Column) 생략 방지
pd.set_option('display.max_columns', None)

def verify_backtest_engine_jupyter():
    # 1. 파라미터 로드 및 인스턴스 초기화
    with open("config/params.yaml", "r", encoding="utf-8") as f:
        params = yaml.safe_load(f)
        
    loader = QuantDataLoader(use_cache=True)
    pipeline = QuantPipeline(params, loader)
    engine = BacktestEngine(pipeline=pipeline, loader=loader)
    
    # 2019년 1분기 ~ 2025년 4분기 말일 생성
    quarter_ends = pd.date_range(start="2019-01-01", end="2025-12-31", freq="Q")

    # date 객체로 변환하여 백테스트 리스트로 사용
    test_dates = [d.date() for d in quarter_ends]

    print(f"총 {len(test_dates)}개의 분기 리밸런싱 시점을 테스트합니다.")

    test_start = test_dates[0]
    test_end = test_dates[-1]
    
    print("=== 백테스트 엔진 검증 시작 ===")
    perf_df, port_df = engine.run(test_start, test_end)
    
    # 3. 주피터 전용 출력 및 파일 저장
    print("\n=== [검증 결과 1] 포트폴리오 구성 로그 (샘플) ===")
    if not port_df.empty:
        display(port_df.head())  # 셀 출력 제한을 피하기 위해 일부만 표시
    else:
        print("포트폴리오 내역이 없습니다.")
    
    print("\n=== [검증 결과 2] 성과 요약 ===")
    if not perf_df.empty:
        display(perf_df)
    else:
        print("성과 내역이 없습니다.")
        
    # 전체 확인을 위한 CSV 추출
    port_df.to_csv("test_portfolio_log.csv", index=False, encoding="utf-8-sig")
    perf_df.to_csv("test_performance_log.csv", index=False, encoding="utf-8-sig")
    print("\n💡 전체 결과는 'test_portfolio_log.csv' 및 'test_performance_log.csv'로 저장되었습니다.")

# 셀 실행
verify_backtest_engine_jupyter()

In [ ]:
import logging
from datetime import date
import yaml
from core.pipeline import QuantPipeline
from data.loader import QuantDataLoader

def check_stage_by_stage_survival_safe():
    # 1. 파이프라인의 수많은 INFO 로그를 가려서 출력 창 폭발 방지
    logging.getLogger().setLevel(logging.WARNING)
    
    with open("config/params.yaml", "r", encoding="utf-8") as f:
        params = yaml.safe_load(f)
        
    # 캐시를 사용하므로 15분 대기 없이 즉각 실행됩니다.
    loader = QuantDataLoader(use_cache=True)
    pipeline = QuantPipeline(params, loader)
    
    test_dates = [date(2026, 7, 31)]
    
    print("🚀 캐시 데이터를 읽어 빠르게 스크리닝을 진행 중입니다...")
    
    # 2. 콘솔 대신 텍스트 파일로 결과만 깔끔하게 저장
    with open("survival_report.txt", "w", encoding="utf-8") as out_f:
        out_f.write("=== 단계별 생존 종목 수 정밀 진단 ===\n")
        
        for bd in test_dates:
            out_f.write(f"\n[기준일: {bd}]\n")
            
            # 파이프라인 실행
            final_df, history = pipeline.run(bd)
            
            # 각 단계별 숫자 기록
            for stage_name, df in history.items():
                out_f.write(f" - {stage_name} 생존: {len(df)}개\n")
                
            out_f.write(f" => 🎯 최종 통과(final_df): {len(final_df)}개\n")
            
    print("✅ 진단 완료! 프로젝트 폴더의 'survival_report.txt' 파일을 열어 확인해 주세요.")

# 셀 실행
check_stage_by_stage_survival_safe()

In [ ]:
import logging
from datetime import date
from data.loader import QuantDataLoader
import pickle

def analyze_stage5_bottleneck_safe():
    # 디버깅 분석을 위해 저장해둔 history 데이터 로드
    try:
        with open('debug_history.pkl', 'rb') as f:
            history_dict= pickle.load(f)
        print("✅ 'debug_history.pkl' 로드 완료! 이제 디버깅을 시작할 수 있습니다.")
    except FileNotFoundError:
        print("❌ 'debug_history.pkl' 파일을 찾을 수 없습니다. 2번 노트북을 먼저 실행해주세요.")

    logging.getLogger().setLevel(logging.WARNING)
    print("🚀 Stage 5 탈락 종목 Raw Data 분석 중입니다 (파일 기록 중)...")

    loader_obj = QuantDataLoader(use_cache=True)
    base_dt = date(2026, 7, 31)
    
    # 1. 딕셔너리 키 및 컬럼명 디버깅 (콘솔에 출력)
    print(f"  [체크] 현재 history에 존재하는 키: {list(history_dict.keys())}")
    
    with open("stage5_bottleneck_report.txt", "w", encoding="utf-8") as out_f:
        out_f.write("==================================================\n")
        out_f.write(f"🩺 [정밀 진단] Stage 5 전원 탈락 사유 분석 (기준일: {base_dt})\n")
        out_f.write("==================================================\n\n")
        
        try:
            # Stage 4 데이터 로드
            stage4_passed = history_dict.get('stage4')
            if stage4_passed is None or stage4_passed.empty:
                print("❌ 에러: Stage 4 데이터가 존재하지 않습니다.")
                return
            
            # 컬럼명 유연성 확보 ('ticker', 'Code', '종목코드' 등)
            ticker_col = 'ticker' if 'ticker' in stage4_passed.columns else stage4_passed.columns[0]
            
            # Stage 5 데이터 로드 (없으면 전원 탈락으로 간주)
            stage5_passed = history_dict.get('stage5')
            if stage5_passed is None:
                out_f.write("⚠️ 'stage5' 데이터가 없습니다. (Stage 4 생존자 전원 탈락으로 간주)\n\n")
                stage5_passed_tickers = []
            else:
                stage5_passed_tickers = stage5_passed[ticker_col].tolist() if not stage5_passed.empty else []
            
            # 탈락자 분류
            stage5_failed_df = stage4_passed[~stage4_passed[ticker_col].isin(stage5_passed_tickers)]
            failed_tickers = stage5_failed_df[ticker_col].tolist()
            
            out_f.write(f"총 {len(failed_tickers)}개 종목이 Stage 5에서 탈락했습니다. 전수 진단 시작...\n\n")
            
            stage5_logs = history_dict.get('stage5_eval_log', None) 
            
            # 2. 로더 직접 호출하여 원인 규명
            for ticker in failed_tickers:
                out_f.write(f"▶️ 대상 종목: {ticker}\n")
                
                # 최근 2분기 시계열 로드
                q_series = loader_obj.get_quarterly_financials_series(ticker, base_dt, n_quarters=2)
                
                if not q_series or len(q_series) < 1:
                    out_f.write("  ❌ [사유] 가용 재무 데이터가 부족하여 분석 불가 (데이터 누락 의심)\n")
                    out_f.write("-" * 50 + "\n")
                    continue
                    
                latest_q = q_series[0]
                op_income = latest_q.get('operating_income', 0)
                int_expense = latest_q.get('interest_expense', 0)
                total_liabilities = latest_q.get('total_liabilities', 0)
                total_equity = latest_q.get('total_equity', 0)
                
                icr = (op_income / int_expense) if int_expense > 0 else float('inf')
                debt_ratio_raw = (total_liabilities / total_equity * 100) if total_equity > 0 else float('inf')
                
                out_f.write(f"  [Raw Data] 영업이익: {op_income:,.0f} | 이자비용: {int_expense:,.0f}\n")
                out_f.write(f"  [Raw Data] 총부채: {total_liabilities:,.0f} | 자기자본: {total_equity:,.0f}\n")
                out_f.write(f"  [지표 산출] 이자보상배율(ICR): {icr:.2f}배 | 절대 부채비율: {debt_ratio_raw:.1f}%\n")
                
                # 로그가 있을 경우 분석
                if stage5_logs is not None and not stage5_logs.empty and ticker in stage5_logs[ticker_col].values:
                    row = stage5_logs[stage5_logs[ticker_col] == ticker].iloc[0]
                    debt_pct = row.get('debt_ratio_percentile', 0)
                    
                    out_f.write(f"  [상대 평가] 섹터 내 부채비율 백분위: {debt_pct:.3f} (컷오프 0.5)\n")
                    
                    if debt_pct > 0.5 and icr < 1.5:
                        out_f.write("  🚨 [사유] 부채비율(하위 50% 밖) & 이자보상배율(1.5 미만) 동시 미달\n")
                    elif debt_pct > 0.5:
                        out_f.write("  🚨 [사유] 부채비율 컷오프 초과 (시설투자/R&D 등 공격적 부채 가능성)\n")
                    elif icr < 1.5:
                        out_f.write("  🚨 [사유] 이자보상배율 미달 (영업이익이 이자비용을 충분히 덮지 못함)\n")
                else:
                    # 로그 없을 경우 절대 수치로 추정
                    if icr < 1.5:
                        out_f.write("  🚨 [추정 사유] 이자보상배율 1.5배 미만 컷오프 탈락\n")
                    else:
                        out_f.write("  ⚠️ [추정 사유] ICR은 양호하나, 부채비율 상대평가(섹터 내 50%)에서 밀린 것으로 보임\n")
                        
                out_f.write("-" * 50 + "\n")
                
        except Exception as e:
            out_f.write(f"\n❌ 진단 중 에러 발생: {e}\n")
            print(f"❌ 에러 발생: {e}")
            
    logging.getLogger().setLevel(logging.INFO)
    print("✅ 진단 완료! 'stage5_bottleneck_report.txt' 파일을 확인해 주세요.")

analyze_stage5_bottleneck_safe()

In [ ]:
import pandas as pd
import numpy as np
import logging
import pickle
from datetime import date
from data.loader import QuantDataLoader
from core import metrics_utils

print("==================================================")
print("🧪 [단독 테스트] Pool 상대평가 기반 Stage 5 가동 (최종 수정본)")
print("==================================================\n")

class FinancialHealthScreenerTest:
    def __init__(self, params: dict):
        self.cutoff_quantile = params.get('stage5_cutoff_quantile', 0.2)
        self.icr_weight = params.get('icr_weight', 0.7)
        self.debt_weight = params.get('debt_weight', 0.3)

    def run(self, input_df: pd.DataFrame, loader, base_date) -> pd.DataFrame:
        if input_df.empty: return input_df
        metrics_data = []

        for _, row in input_df.iterrows():
            ticker = row['ticker']
            sector = row.get('sector', 'N/A')
            
            raw_ttm = loader.get_ttm_financials(ticker, base_date)

            total_liab = raw_ttm.get('total_liabilities', np.nan)
            total_equity = raw_ttm.get('total_equity', np.nan)
            debt_ratio = np.divide(total_liab, total_equity) if pd.notna(total_liab) and pd.notna(total_equity) else np.nan

            ocf = raw_ttm.get('operating_cash_flow', np.nan)
            ni = raw_ttm.get('net_income', np.nan)
            op_inc = raw_ttm.get('operating_income', np.nan)
            int_exp = raw_ttm.get('interest_expense', np.nan)

            warning_tags = []
            na_reasons = []
            icr = np.nan

            if pd.notna(op_inc) and pd.notna(int_exp):
                if int_exp <= 0:
                    icr = 50.0  
                else:
                    icr = np.divide(op_inc, int_exp)

            if any(keyword in str(sector) for keyword in ['금융', '증권', '보험', '은행', '지주']):
                na_reasons.append(metrics_utils.MetricStatus.CAUTION)

            metrics_data.append({
                'ticker': ticker,
                'sector': sector,
                'debt_ratio': debt_ratio,
                'ocf': ocf,
                'net_income': ni,
                'interest_coverage_ratio': icr,
                'na_reasons': ",".join(na_reasons),
                'warning_tags': ""
            })

        df = pd.DataFrame(metrics_data)

        # 캡핑 및 태그 부착
        df['icr_capped'] = df['interest_coverage_ratio'].clip(lower=-10, upper=50)
        df['debt_ratio_capped'] = df['debt_ratio'].clip(upper=5)

        df.loc[df['interest_coverage_ratio'] < 1.0, 'warning_tags'] += "[ICR미달]"
        df.loc[(df['ocf'] < df['net_income'].fillna(-np.inf)), 'warning_tags'] += "[이익질주의]"
        
        # 부채비율 랭크는 섹터 내 비교를 유지해도 무방 (태그용이므로)
        df['debt_rank_pct'] = df.groupby('sector')['debt_ratio'].rank(pct=True, ascending=True)
        cond_not_finance = ~df['na_reasons'].astype(str).str.contains(metrics_utils.MetricStatus.CAUTION)
        df.loc[cond_not_finance & (df['debt_rank_pct'] > 0.90), 'warning_tags'] += "[과다부채]"

        # 🚨 [핵심 수정 구간] 🚨 
        # Z-score 산출 시 .groupby('sector')를 완전히 제거하고 14개 종목 Pool 전체를 대상으로 계산
        df['debt_ratio_inv'] = -df['debt_ratio_capped']
        df['icr_zscore'] = metrics_utils.calc_zscore(df['icr_capped'])
        df['debt_zscore'] = metrics_utils.calc_zscore(df['debt_ratio_inv'])

        score_mapping = {'icr_zscore': self.icr_weight, 'debt_zscore': self.debt_weight}
        df['stage5_score'] = metrics_utils.compute_composite_score(df, score_mapping)

        cond_finance = df['na_reasons'].str.contains(metrics_utils.MetricStatus.CAUTION)
        df.loc[cond_finance, 'stage5_score'] = df.loc[cond_finance, 'icr_zscore']

        # 컷오프
        top_percentile = 1.0 - self.cutoff_quantile 
        if len(df) > 5:
            passed_df, _ = metrics_utils.apply_percentile_filter(df, 'stage5_score', top_percentile)
        else:
            passed_df = df.copy()

        return passed_df, df 

# ---------------------------------------------------------
# 2. History 로드 및 실행부
# ---------------------------------------------------------
def analyze_stage5_bottleneck_safe():
    try:
        with open('debug_history.pkl', 'rb') as f:
            history_dict = pickle.load(f)
        
        stage4_df = history_dict.get('stage4')
        if stage4_df is None or stage4_df.empty:
            print("❌ Stage 4 데이터가 비어있습니다. 파이프라인 이력을 확인하세요.")
            return

    except FileNotFoundError:
        print("❌ 'debug_history.pkl' 파일을 찾을 수 없습니다.")
        return

    logging.getLogger().setLevel(logging.WARNING)

    loader_obj = QuantDataLoader(use_cache=True)
    base_dt = date(2026, 7, 31)

    test_params = {
        'stage5_cutoff_quantile': 0.2,
        'icr_weight': 0.7,
        'debt_weight': 0.3
    }
    
    screener = FinancialHealthScreenerTest(test_params)
    passed_df, full_df = screener.run(stage4_df, loader_obj, base_dt)

    print(f"🎯 투입(Stage 4): {len(stage4_df)}개 -> 생존(Stage 5): {len(passed_df)}개")
    
    cols_to_show = ['ticker', 'interest_coverage_ratio', 'debt_ratio', 'icr_zscore', 'debt_zscore', 'stage5_score', 'warning_tags']
    
    print("\n▶️ [생존 종목 Top 5 (점수 순)]")
    print(passed_df.sort_values('stage5_score', ascending=False)[cols_to_show].head().to_string(index=False))
    
    failed_df = full_df[~full_df['ticker'].isin(passed_df['ticker'])]
    if not failed_df.empty:
        print(f"\n▶️ [탈락 종목 명단 ({len(failed_df)}개)]")
        print(failed_df[cols_to_show].sort_values('stage5_score').to_string(index=False))

# 실행
analyze_stage5_bottleneck_safe()